# Feature Engineering + XGBoost

In this notebook, I will test whether useful relationships between the original features can improve the XGBoost model.

The goal is to create features that make sense for EV purchasing, while keeping the original features as well.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

In [2]:
train = pd.read_csv('../data/train.csv')

X = train.drop(columns=['Will_Buy_EV', 'id']).copy()
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training shape:', X_train.shape)
print('Validation shape:', X_valid.shape)

Training shape: (534932, 13)
Validation shape: (133733, 13)


## Create Features

These features describe relationships that may matter when someone considers buying an EV, such as commute distance, charging access, income, and range anxiety.

In [3]:
def add_features(df):
    df = df.copy()

    df['Total_Charging_Stations'] = (
        df['Charging_Stations_Near_Home'] +
        df['Charging_Stations_Near_Work']
    )

    df['Charging_Station_Difference'] = (
        df['Charging_Stations_Near_Work'] -
        df['Charging_Stations_Near_Home']
    )

    df['Charging_per_Commute_km'] = (
        df['Total_Charging_Stations'] /
        (df['Daily_Commute_km'] + 1)
    )

    df['Commute_per_Charging'] = (
        df['Daily_Commute_km'] /
        (df['Total_Charging_Stations'] + 1)
    )

    range_map = {'Low': 1, 'Medium': 2, 'High': 3}
    df['Range_Anxiety_Score'] = df['Range_Anxiety_Level'].map(range_map)

    df['Commute_Range_Concern'] = (
        df['Daily_Commute_km'] * df['Range_Anxiety_Score']
    )

    df['Income_Environmental_Concern'] = (
        df['Annual_Income_USD'] *
        df['Environmental_Concern_Level']
    )

    df['Income_per_Car'] = (
        df['Annual_Income_USD'] /
        df['Number_of_Cars_Owned']
    )

    home_charge_map = {'No': 0, 'Yes': 1}
    df['Home_Charging_Score'] = df['Home_Charging_Possible'].map(home_charge_map)

    df['Home_Charging_Total_Access'] = (
        df['Home_Charging_Score'] *
        df['Total_Charging_Stations']
    )

    df['Subsidy_Environmental_Concern'] = (
        (df['Subsidy_Available'] == 'Yes').astype(int) *
        df['Environmental_Concern_Level']
    )

    df['Cars_Commute_Interaction'] = (
        df['Number_of_Cars_Owned'] *
        df['Daily_Commute_km']
    )

    return df

X_train_fe = add_features(X_train)
X_valid_fe = add_features(X_valid)

print('Original features:', X_train.shape[1])
print('Features after engineering:', X_train_fe.shape[1])

print('\nNew features:')
print([col for col in X_train_fe.columns if col not in X_train.columns])

Original features: 13
Features after engineering: 25

New features:
['Total_Charging_Stations', 'Charging_Station_Difference', 'Charging_per_Commute_km', 'Commute_per_Charging', 'Range_Anxiety_Score', 'Commute_Range_Concern', 'Income_Environmental_Concern', 'Income_per_Car', 'Home_Charging_Score', 'Home_Charging_Total_Access', 'Subsidy_Environmental_Concern', 'Cars_Commute_Interaction']


In [4]:
numeric_features = X_train_fe.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_fe.select_dtypes(include=['object']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

C:\Users\aakif\AppData\Local\Temp\ipykernel_16628\500793404.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train_fe.select_dtypes(include=['object']).columns.tolist()


In [5]:
pipeline.fit(X_train_fe, y_train)

valid_predictions = pipeline.predict_proba(X_valid_fe)[:, 1]
auc = roc_auc_score(y_valid, valid_predictions)

print(f'Feature Engineering + XGBoost ROC-AUC: {auc:.6f}')

Feature Engineering + XGBoost ROC-AUC: 0.941344


## Comparison

The previous XGBoost model scored **0.941635** on the same validation split.

If this experiment improves the validation score, we can use the feature-engineered version for Submission 03.